# Lecture 8.2: Your First Custom Tool

**Section 08: MCP, Connecting External Systems**

In this notebook you give a Claude agent its first custom tool. The tool is a plain Python function from Pixel Forge, a two-person indie game studio, wrapped so the agent can call it through an in-process MCP server.

By the end you will have:

| Step | What you build | Cells |
|---|---|---|
| Ground truth | A plain Python function that calculates XP per level | 4 |
| Tool | The same function wrapped with `@tool` | 5 |
| Server | An in-process MCP server that holds the tool | 6 |
| Discovery | Proof that the agent can see the tool, under its full name | 7 |
| Agent run | An agent that calls the tool to answer a question | 8 to 10 |
| Second tool | A second tool on the same server, approved with a wildcard | 11 |
| Approval | The same question with the tool visible but not approved | 12 |

Run the cells in order, top to bottom. Each one depends on the cells above it.

> **Starter notebook.** Cells 5, 6 and 8 are empty on purpose. Type them as you follow along with the video. The solution notebook has every cell filled in.

## Cell 1: Install the Claude Agent SDK

This cell installs the Claude Agent SDK, the Python package that runs Claude as an agent. It is the only install this notebook needs. The package also bundles the Claude Code CLI, which runs the agent loop behind the scenes, so there is nothing else to set up.

The version is pinned to `0.2.188` so the notebook behaves the way it does in the video. The comment in the cell explains how to install the latest version instead. Newer versions may behave differently.

The `-q` flag keeps the install output quiet. If the package is already installed in this runtime, pip confirms that quickly and moves on.

In [ ]:
# We pin the Claude Agent SDK to a specific version so every example runs
# exactly as shown in the course. To use the latest version, run:
# pip install -U claude-agent-sdk
# Or substitute your preferred version below. Newer versions may behave
# differently from the videos.
!pip install claude-agent-sdk==0.2.188 -q

## Cell 2: Load your API key from Colab Secrets

The agent needs your Anthropic API key. This notebook reads it from Colab Secrets, so the key never appears in the notebook itself.

If you have not added the secret yet:

1. Click the **key icon** in the left sidebar of Colab.
2. Click **Add new secret**.
3. Name it exactly `ANTHROPIC_API_KEY`.
4. Paste your key into the value field.
5. Switch **Notebook access** on.

You create the key in the Claude Console at https://platform.claude.com/ under **API Keys**.

The cell copies the secret into the `ANTHROPIC_API_KEY` environment variable, which is where the SDK looks for it.

> **Running locally instead of in Colab?** Skip this cell and set the `ANTHROPIC_API_KEY` environment variable in your terminal before you start Jupyter.

In [ ]:
from google.colab import userdata
import os

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

## Cell 3: Choose the model

`MODEL_NAME` holds the model every agent in this notebook uses. Each `ClaudeAgentOptions` below passes `model=MODEL_NAME`, so changing this one line changes the model for the whole notebook.

The default is Claude Haiku 5.5. Model names change with new releases, so check the models page linked in the comment for the current list.

In [ ]:
# See latest models at: https://platform.claude.com/docs/en/about-claude/models/overview
MODEL_NAME = "claude-haiku-5-5"

## Cell 4: Meet Pixel Forge

Pixel Forge is a two-person indie game studio, and it is the running example for this section. In their game, players earn experience points (XP) by defeating enemies and completing quests, and they need a set amount of XP to reach each new level.

The studio's rule is that each level needs 20% more XP than the one before, starting at 100 XP for level 1:

```
xp_for_level(level) = round(100 * 1.2 ** (level - 1))
```

This cell is plain Python with no agent involved. It is the **ground truth**: the exact numbers every later cell is checked against.

| Level | XP needed |
|---|---|
| 1 | 100 |
| 5 | 207 |
| 10 | 516 |

**Why put this in a tool?** A language model can attempt arithmetic like `1.2 ** 9` in its head, but it predicts digits rather than computing them, so it can be slightly off. A tool hands the calculation to Python, which gets it right every time. The agent decides *when* to use the tool, and the tool supplies the *exact* number.

In [ ]:
def xp_for_level(level):
    """XP a player needs to reach the given level in Pixel Forge's game."""
    return round(100 * 1.2 ** (level - 1))


for level in (1, 5, 10):
    print(f"Level {level}: {xp_for_level(level)} XP")

assert xp_for_level(1) == 100
assert xp_for_level(5) == 207
assert xp_for_level(10) == 516
print("Ground truth checks passed.")

## Cell 5: Wrap it as a tool

The `@tool` decorator turns an async Python function into a tool an agent can call. It takes three arguments:

| Argument | Value here | What it is for |
|---|---|---|
| `name` | `"xp_for_level"` | The identifier the agent uses to call the tool. |
| `description` | A sentence about XP per level | What the agent reads to decide **when** to call the tool. Make it specific. |
| `input_schema` | `{"level": int}` | The arguments the agent must supply. |

**The input schema.** `{"level": int}` is the simple form: a dict that maps each argument name to a Python type. The SDK converts it to JSON Schema for you, and in this form every key is required. When you need optional fields, enums or ranges, you can pass a full JSON Schema dict instead. Arguments are validated against the schema *before* your function runs. If the agent sends something invalid, such as text where an integer belongs, it gets an error result back and your function never runs.

**The handler.** The decorated function must be `async`. It receives one dict of validated arguments and returns a dict with a `content` list of result blocks. A text block looks like `{"type": "text", "text": "..."}`, and that text is what the agent reads as the tool's result.

The decorator replaces the function with an `SdkMcpTool` object. The cell prints its type and its `name` to show that.

In [ ]:
# LIVE CODE - type during recording

## Cell 6: Create the server

Agents reach custom tools through MCP, so a tool has to live on an MCP server. `create_sdk_mcp_server()` builds one that runs **in-process**: inside this notebook's Python process, with no separate program to start or manage.

| Argument | Value here | Meaning |
|---|---|---|
| `name` | `"balance"` | The server's name. Pixel Forge's game balance numbers live here. |
| `tools` | `[xp_tool]` | The tools on this server. A server can hold as many as you list. |
| `version` | not set | Optional, defaults to `"1.0.0"`. Informational only. |

The return value is a server config: a plain dict with a `type` of `"sdk"`, the `name`, and the live server `instance`. Because it is a dict, you read its fields with square brackets. Nothing talks to Claude yet. The config is handed to an agent in the next cell.

In [ ]:
# LIVE CODE - type during recording

## Cell 7: See the tool appear

Before asking a real question, this cell checks what the agent can see. It passes the server in `mcp_servers` and gives a trivial prompt, so the run is short.

**The init message.** The first message `query()` streams back is a `SystemMessage` with `subtype == "init"`. Its `data["tools"]` lists every tool available in the session. MCP tools start with `mcp__`, so the cell filters on that prefix. It also prints the status of each MCP server from `data["mcp_servers"]`, which should read `connected`.

**The naming pattern.** Every MCP tool gets a full name:

```
mcp__{server_name}__{tool_name}
```

Expect `mcp__balance__xp_for_level`. The `{server_name}` part comes from the **key** you use in the `mcp_servers` dictionary, not from the `name` you passed to `create_sdk_mcp_server()`. This notebook uses `"balance"` for both, so they match, but the key is what counts.

No tools are approved in this cell, because nothing needs to run. The agent can see the tool without being allowed to call it.

In [ ]:
from claude_agent_sdk import query, ClaudeAgentOptions, SystemMessage, ResultMessage

async for message in query(
    prompt="Reply with the single word: ready",
    options=ClaudeAgentOptions(
        mcp_servers={"balance": balance_server},
        model=MODEL_NAME,
        cwd="/content",
    ),
):
    if isinstance(message, SystemMessage) and message.subtype == "init":
        mcp_tools = [t for t in message.data.get("tools", []) if t.startswith("mcp__")]
        print("MCP tools the agent can see:", mcp_tools)
        for server in message.data.get("mcp_servers", []):
            print("Server status:", server.get("name"), server.get("status"))
    if isinstance(message, ResultMessage):
        print("Agent replied:", message.result)

## Cell 8: Run the agent

Now the agent gets a question it should answer with the tool. Two options work together here, and they are two separate steps:

| Option | Step | What it does |
|---|---|---|
| `mcp_servers={"balance": balance_server}` | Make available | Connects the server, so the agent can see its tools. |
| `allowed_tools=["mcp__balance__xp_for_level"]` | Approve | Lets the agent call that tool without a permission prompt. |

MCP tools need explicit approval. Without it, the agent can see a tool but cannot call it. `allowed_tools` matches the **full** name, `mcp__balance__xp_for_level`, because that is the name the agent calls the tool by. Only the one tool this demo needs is approved.

`model=MODEL_NAME` and `cwd="/content"` are set as in every agent call in this course.

Remember that `query()` does not return an answer. It streams messages while the agent works: tool calls, tool results, and finally a `ResultMessage` holding the answer. The cell keeps every message in `msgs` so the next cell can look inside, and prints the final answer.

The agent's wording changes from run to run. The number should not: look for **516**.

In [ ]:
# LIVE CODE - type during recording

## Cell 9: Inspect the stream

The answer looked right, but did it come from the tool? This cell looks inside `msgs` to find out.

| Message | Block | What it holds |
|---|---|---|
| `AssistantMessage` | `ToolUseBlock` | The call: `name`, `input` and a unique `id`. |
| `UserMessage` | `ToolResultBlock` | The result, with `tool_use_id` equal to the call's `id`. |
| `ResultMessage` | none | The final answer in `result`. |

Tool results come back inside a `UserMessage`, not an `AssistantMessage`, because from the model's point of view a tool result is new input. The `id` and `tool_use_id` are how a call and its result are matched.

The cell defines three small helpers that later cells reuse:

* `tool_calls_in(msgs, prefix)` collects every `ToolUseBlock` whose name starts with `prefix`.
* `result_for(msgs, tool_use_id)` returns the text of the matching `ToolResultBlock`.
* `final_result(msgs)` returns the `ResultMessage`.

You may also see a `ToolSearch` call in the stream. Tool search is on by default, and the agent can use it to load a tool's full definition before calling it. The helpers filter by name, so it does not affect the checks.

In [ ]:
from claude_agent_sdk import AssistantMessage, UserMessage, ToolUseBlock, ToolResultBlock


def tool_calls_in(msgs, prefix="mcp__balance__"):
    """Every ToolUseBlock whose name starts with prefix."""
    calls = []
    for m in msgs:
        if isinstance(m, AssistantMessage):
            for block in m.content:
                if isinstance(block, ToolUseBlock) and block.name.startswith(prefix):
                    calls.append(block)
    return calls


def result_for(msgs, tool_use_id):
    """Text of the ToolResultBlock that answers one tool call."""
    for m in msgs:
        if isinstance(m, UserMessage) and isinstance(m.content, list):
            for block in m.content:
                if isinstance(block, ToolResultBlock) and block.tool_use_id == tool_use_id:
                    if isinstance(block.content, str):
                        return block.content
                    return " ".join(p.get("text", "") for p in block.content or [])
    return None


def final_result(msgs):
    """The ResultMessage at the end of the stream."""
    for m in msgs:
        if isinstance(m, ResultMessage):
            return m
    return None


for call in tool_calls_in(msgs, "mcp__balance__xp_for_level"):
    print("Tool called:", call.name)
    print("Input:      ", call.input)
    print("Result:     ", result_for(msgs, call.id))

answer = final_result(msgs).result or ""
print()
print("516 appears in the final answer:", "516" in answer)

## Cell 10: A question needing several calls

The tool answers one level at a time. This question covers twelve levels, so the agent has to decide how to use the tool. Nothing in the code tells it what to do. It plans the calls itself.

The cell reuses `options` from Cell 8, counts the `xp_for_level` calls, prints each input and result, and checks the final answer against the ground truth from Cell 4:

```
sum(xp_for_level(n) for n in range(1, 13))  # 3958
```

Two outcomes are possible, and both teach something:

* **Twelve calls.** Every number came from the tool, and the agent added them up.
* **Fewer calls.** The agent filled in some levels itself, or added up the numbers in its head. That arithmetic is where errors creep in, which is exactly why exact work belongs in tools.

The final check accepts the total written with or without a thousands separator. Compare the count and the total with what you see on screen, because they can change from run to run.

In [ ]:
msgs_total = []
async for message in query(
    prompt="What is the total XP needed across levels 1 to 12?",
    options=options,
):
    msgs_total.append(message)

calls = tool_calls_in(msgs_total, "mcp__balance__xp_for_level")
print("xp_for_level calls:", len(calls))
for call in calls:
    print("  ", call.input, "->", result_for(msgs_total, call.id))

expected = sum(xp_for_level(n) for n in range(1, 13))
answer = final_result(msgs_total).result or ""
print()
print(answer)
print()
print("Expected total:", expected)
print("Exact total in the answer:", str(expected) in answer or f"{expected:,}" in answer)
if len(calls) < 12:
    print("The agent made fewer than 12 calls, so some of this answer came from its own reasoning.")

## Cell 11: A second tool

Pixel Forge also wants to know how many hits it takes to defeat an enemy. This cell adds a second tool, `hits_to_defeat`, to the same server.

| Piece | Detail |
|---|---|
| Schema | `{"damage": int, "enemy_health": int}`. Both are required. |
| Formula | `math.ceil(enemy_health / damage)`. A partial hit still counts as a hit. |
| Error result | Damage of 0 or less cannot defeat anything, so the handler returns `"is_error": True` with a message the agent can read. |

**Rebuilding the server.** A server's tools are fixed when it is created, so the cell calls `create_sdk_mcp_server()` again with both tools and replaces `balance_server`.

**Wildcard approval.** Instead of listing each tool, `allowed_tools=["mcp__balance__*"]` approves every tool on the `balance` server. It is still least privilege: one server, nothing else.

The question gives a sword with 12 damage and an enemy with 85 health. The answer should be **8**, because 7 hits deal only 84 damage. The cell prints which tool the agent picked. The descriptions are what let it choose correctly.

In [ ]:
import math
import re


@tool(
    "hits_to_defeat",
    "Get how many hits it takes to defeat an enemy, given damage per hit and enemy health.",
    {"damage": int, "enemy_health": int},
)
async def hits_tool(args):
    damage = args["damage"]
    enemy_health = args["enemy_health"]
    if damage <= 0:
        return {
            "content": [
                {"type": "text", "text": f"Damage must be greater than 0, got {damage}."}
            ],
            "is_error": True,
        }
    hits = math.ceil(enemy_health / damage)
    return {
        "content": [
            {"type": "text", "text": f"{hits} hits of {damage} damage defeat an enemy with {enemy_health} health."}
        ]
    }


balance_server = create_sdk_mcp_server(
    name="balance",
    tools=[xp_tool, hits_tool],
)

msgs_hits = []
async for message in query(
    prompt="A sword deals 12 damage. How many hits to defeat an enemy with 85 health?",
    options=ClaudeAgentOptions(
        mcp_servers={"balance": balance_server},
        allowed_tools=["mcp__balance__*"],
        model=MODEL_NAME,
        cwd="/content",
    ),
):
    msgs_hits.append(message)

for call in tool_calls_in(msgs_hits):
    print("Tool called:", call.name, call.input)
    print("Result:     ", result_for(msgs_hits, call.id))

answer = final_result(msgs_hits).result or ""
print()
print(answer)
print()
print("Expected:", math.ceil(85 / 12))
print("8 appears in the final answer:", re.search(r"\b8\b", answer) is not None)

## Cell 12: Approval matters

This cell asks Cell 8's question again with one change: `allowed_tools=[]`. The server is still connected, so the tool is still visible, just as it was in Cell 7. Nothing is approved.

The cell prints three things:

| Output | What it tells you |
|---|---|
| Tool call attempts | Whether the agent tried to call `xp_for_level` at all. |
| `permission_denials` | A list on the `ResultMessage` of tool calls that were refused, if any. |
| The final answer | What the agent said without the tool. |

Compare it with Cell 8. Being visible and being allowed to run are two different things. If the agent still gives a number here, it worked it out on its own, so check it against **516** from Cell 4.

In [ ]:
msgs_denied = []
async for message in query(
    prompt="How much XP does a player need to reach level 10?",
    options=ClaudeAgentOptions(
        mcp_servers={"balance": balance_server},
        allowed_tools=[],
        model=MODEL_NAME,
        cwd="/content",
    ),
):
    msgs_denied.append(message)

attempts = tool_calls_in(msgs_denied, "mcp__balance__xp_for_level")
print("Tool call attempts:", len(attempts))
for call in attempts:
    print("  Result:", result_for(msgs_denied, call.id))

result = final_result(msgs_denied)
print("Permission denials:", result.permission_denials)
print()
print(result.result)